In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold

In [2]:
# Load data
df = pd.read_csv("stock_prices.csv")

# Keep only the required columns
keep_cols = ["RowId", "Date", "SecuritiesCode", "Open", "High", "Low", "Close", "Volume", "Target"]
df = df[keep_cols].copy()

# Parse Date as datetime (needed for a proper time-based split)
df["Date"] = pd.to_datetime(df["Date"])

# Sort chronologically — critical for a time split
df = df.sort_values("Date").reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

# ---- Time-based train/test split ----
# Split by date so that all rows of a given day stay in the same split
# (avoids look-ahead / leakage from the same trading day).
split_date = df["Date"].quantile(0.8)  # 80% of dates for training

train = df[df["Date"] <= split_date].copy()
test  = df[df["Date"] >  split_date].copy()

print(f"\nTrain: {train['Date'].min().date()} → {train['Date'].max().date()}  ({len(train)} rows)")
print(f"Test : {test['Date'].min().date()} → {test['Date'].max().date()}  ({len(test)} rows)")

# Optional: save the splits
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

Total rows: 2332531
Date range: 2017-01-04 → 2021-12-03

Train: 2017-01-04 → 2020-12-21  (1866532 rows)
Test : 2020-12-22 → 2021-12-03  (465999 rows)


# Missing Value Diagnostic Script

In [3]:
# In a notebook, use CWD (should be the folder with the notebook)
BASE_DIR = Path.cwd()
print("Working dir:", BASE_DIR)
print("Files here :", [f.name for f in BASE_DIR.iterdir() if f.is_file()])

# Continue from the split: load the train set
train = pd.read_csv(BASE_DIR / "train.csv")
test  = pd.read_csv(BASE_DIR / "test.csv")

# Re-parse Date (CSV round-trip turns it back into string)
train["Date"] = pd.to_datetime(train["Date"])
test["Date"]  = pd.to_datetime(test["Date"])

print(f"\ntrain: {train.shape}")
print(f"test : {test.shape}")
train.head()

Working dir: d:\Perkuliahan\Sems 5\Data Sains\Datsai_Kel-1\EDA&Preprocessing
Files here : ['Main_Notebook.ipynb', 'stock_prices.csv', 'test.csv', 'train.csv']

train: (1866532, 9)
test : (465999, 9)


,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,Target
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,0.000730
1,20170104_7412,2017-01-04,7412,719.0,725.0,719.0,721.0,201400,0.000000
2,20170104_7408,2017-01-04,7408,2459.0,2518.0,2447.0,2500.0,110900,0.004421
3,20170104_7315,2017-01-04,7315,465.0,494.0,465.0,493.0,41100,-0.004032
4,20170104_7313,2017-01-04,7313,3055.0,3150.0,3045.0,3135.0,248600,-0.009693


In [ ]:

print("=" * 70)
print("1. BASIC SHAPE & DTYPES")
print("=" * 70)
print(f"Shape: {train.shape}")
print("\nDtypes:")
print(train.dtypes)

print("\n" + "=" * 70)
print("2. MISSING VALUE COUNT & PERCENTAGE PER COLUMN  (train)")
print("=" * 70)
missing = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct":   (train.isna().mean() * 100).round(4),
    "dtype":         train.dtypes.astype(str),
}).sort_values("missing_count", ascending=False)
print(missing)

print("\n--- same check on test ---")
missing_test = pd.DataFrame({
    "missing_count": test.isna().sum(),
    "missing_pct":   (test.isna().mean() * 100).round(4),
}).sort_values("missing_count", ascending=False)
print(missing_test)

print("\n" + "=" * 70)
print("3. TOTAL MISSING CELLS & ROWS AFFECTED  (train)")
print("=" * 70)
total_cells           = train.shape[0] * train.shape[1]
total_missing         = train.isna().sum().sum()
rows_with_any_missing = train.isna().any(axis=1).sum()
rows_with_all_missing = train.isna().all(axis=1).sum()
print(f"Total cells           : {total_cells}")
print(f"Total missing cells   : {total_missing}  ({total_missing/total_cells*100:.4f}%)")
print(f"Rows w/ >=1 missing   : {rows_with_any_missing}  ({rows_with_any_missing/len(train)*100:.4f}%)")
print(f"Rows w/ ALL missing   : {rows_with_all_missing}")

print("\n" + "=" * 70)
print("4. MISSING VALUE POSITION PER COLUMN  (train)")
print("=" * 70)
for col in train.columns:
    if train[col].isna().any():
        idx = train.index[train[col].isna()]
        print(f"{col:16s} | first @row {idx.min():>7} | last @row {idx.max():>7} | n_missing={len(idx)}")
    else:
        print(f"{col:16s} | no missing values")

print("\n" + "=" * 70)
print("5. MISSING VALUES BY YEAR  (train) — is the gap temporal?")
print("=" * 70)
train["_year"] = train["Date"].dt.year
print(train.groupby("_year").apply(lambda g: g.isna().sum()))
train.drop(columns="_year", inplace=True)

print("\n" + "=" * 70)
print("6. MISSING VALUES BY SECURITY  (train) — is the gap per-ticker?")
print("=" * 70)
per_sec = train.groupby("SecuritiesCode").apply(lambda g: g.isna().sum())
per_sec = per_sec[per_sec.sum(axis=1) > 0]
if per_sec.empty:
    print("No security has any missing values.")
else:
    print(f"{per_sec.shape[0]} securities have missing values. Top 10:")
    print(per_sec.assign(total=per_sec.sum(axis=1))
                  .sort_values("total", ascending=False).head(10))

print("\n" + "=" * 70)
print("7. NUMERIC DISTRIBUTION  (train)")
print("=" * 70)
num_cols = ["Open", "High", "Low", "Close", "Volume", "Target"]
print(train[num_cols].describe().T)

print("\n" + "=" * 70)
print("8. SKEWNESS  (train) — mean vs median decision")
print("=" * 70)
print(train[num_cols].skew().round(4))

print("\n" + "=" * 70)
print("9. SAMPLE ROWS WITH ANY MISSING VALUE  (train)")
print("=" * 70)
sample = train[train.isna().any(axis=1)].head(10)
print("No rows contain missing values." if sample.empty else sample.to_string())

print("\n" + "=" * 70)
print("10. MAX CONSECUTIVE MISSING RUN PER COLUMN  (train)")
print("=" * 70)
def max_consecutive_na(s):
    isna = s.isna().to_numpy()
    if not isna.any():
        return 0
    max_run = cur = 0
    for v in isna:
        cur = cur + 1 if v else 0
        max_run = max(max_run, cur)
    return max_run

for col in train.columns:
    print(f"{col:16s} | max consecutive missing run: {max_consecutive_na(train[col])}")

# Implementation — fit-on-train, apply-to-test

In [5]:
# ---------------------------------------------------------------
# Confirm the "no trading day" signature before dropping
# (Open/High/Low/Close all NaN  AND  Volume == 0)
# ---------------------------------------------------------------
ohlc_cols = ["Open", "High", "Low", "Close"]

for name, df_ in [("train", train), ("test", test)]:
    mask = df_[ohlc_cols].isna().any(axis=1)
    n = mask.sum()
    n_zero_vol = (df_.loc[mask, "Volume"] == 0).sum()
    print(f"{name}: OHLC-missing rows = {n}, of which Volume==0 = {n_zero_vol} "
          f"({n_zero_vol/n*100:.2f}%)" if n else f"{name}: no OHLC-missing rows")

# ---------------------------------------------------------------
# Drop rules (applied identically to train and test):
#   1) rows with any missing OHLC  -> non-trading days
#   2) rows with missing Target    -> unusable label
# ---------------------------------------------------------------
drop_subset = ["Open", "High", "Low", "Close", "Target"]

train_clean = train.dropna(subset=drop_subset).reset_index(drop=True)
test_clean  = test.dropna(subset=drop_subset).reset_index(drop=True)

print(f"\nBefore: train={train.shape}  test={test.shape}")
print(f"After : train={train_clean.shape}  test={test_clean.shape}")
print(f"Dropped train rows: {len(train) - len(train_clean)} "
      f"({(len(train)-len(train_clean))/len(train)*100:.4f}%)")
print(f"Dropped test  rows: {len(test)  - len(test_clean)} "
      f"({(len(test)-len(test_clean))/len(test)*100:.4f}%)")

# ---------------------------------------------------------------
# Post-cleanup verification
# ---------------------------------------------------------------
print("\nRemaining missing values — train:", int(train_clean.isna().sum().sum()))
print("Remaining missing values — test :", int(test_clean.isna().sum().sum()))

# Any security fully wiped out?
before_secs = set(train["SecuritiesCode"].unique())
after_secs  = set(train_clean["SecuritiesCode"].unique())
lost = before_secs - after_secs
print(f"Securities present in train before: {len(before_secs)}  |  after: {len(after_secs)}")
if lost:
    print(f"  ⚠ {len(lost)} securities fully removed: {sorted(lost)[:20]}{'...' if len(lost) > 20 else ''}")
else:
    print("  ✓ No security was completely removed.")

# Sanity: Volume should now be > 0 wherever OHLC exists (mostly)
print("\nVolume==0 rows remaining in train_clean:", int((train_clean["Volume"] == 0).sum()))
print("Volume==0 rows remaining in test_clean :", int((test_clean["Volume"]  == 0).sum()))

train: OHLC-missing rows = 6503, of which Volume==0 = 6503 (100.00%)
test: OHLC-missing rows = 1105, of which Volume==0 = 1105 (100.00%)

Before: train=(1866532, 9)  test=(465999, 9)
After : train=(1860029, 9)  test=(464894, 9)
Dropped train rows: 6503 (0.3484%)
Dropped test  rows: 1105 (0.2371%)

Remaining missing values — train: 0
Remaining missing values — test : 0
Securities present in train before: 1997  |  after: 1997
  ✓ No security was completely removed.

Volume==0 rows remaining in train_clean: 0
Volume==0 rows remaining in test_clean : 0


# Encoding Diagnostic Script


In [8]:
import pandas as pd
import numpy as np

# Use the in-memory dataframes from the missing-value step
train = train_clean
test  = test_clean

print("=" * 70)
print("0. CONFIRM WE'RE ON THE CLEANED DATA")
print("=" * 70)
print(f"train shape : {train.shape}")
print(f"test shape  : {test.shape}")
print(f"train NaN   : {int(train.isna().sum().sum())}")
print(f"test  NaN   : {int(test.isna().sum().sum())}")

print("\n" + "=" * 70)
print("1. COLUMN DTYPES & NON-NULL COUNTS  (train)")
print("=" * 70)
info = pd.DataFrame({
    "dtype":        train.dtypes.astype(str),
    "n_unique":     train.nunique(dropna=False),
    "n_rows":       len(train),
    "unique_ratio": (train.nunique(dropna=False) / len(train)).round(6),
})
print(info)

print("\n" + "=" * 70)
print("2. WHICH COLUMNS ARE CATEGORICAL CANDIDATES?")
print("=" * 70)
for col in train.columns:
    dt    = train[col].dtype
    n     = train[col].nunique(dropna=False)
    ratio = n / len(train)
    is_cat_dtype = str(dt) in ("object", "string", "str", "category")
    tag = []
    if is_cat_dtype:
        tag.append("dtype=categorical")
    if ratio > 0.95 and not str(dt).startswith("datetime"):
        tag.append("IDENTIFIER (nearly unique)")
    if 1 < n <= 50:
        tag.append("low-cardinality")
    print(f"{col:16s} | dtype={str(dt):15s} | nunique={n:>8} | ratio={ratio:.4f} | "
          f"{' , '.join(tag) if tag else '-'}")

print("\n" + "=" * 70)
print("3. DETAILED LOOK AT OBJECT / STRING COLUMNS")
print("=" * 70)
obj_cols = train.select_dtypes(include=["object", "string", "str", "category"]).columns.tolist()
if not obj_cols:
    print("No object/string/category columns found.")
else:
    for col in obj_cols:
        print(f"\n--- {col} ---")
        print(f"dtype        : {train[col].dtype}")
        print(f"n_unique     : {train[col].nunique()}")
        print(f"top 5 values : {train[col].value_counts().head(5).to_dict()}")
        print(f"sample       : {train[col].dropna().unique()[:5]}")

print("\n" + "=" * 70)
print("4. STRUCTURE OF `RowId`  (is it unique / composite?)")
print("=" * 70)
print(f"n_unique RowId       : {train['RowId'].nunique()}")
print(f"n_rows               : {len(train)}")
print(f"RowId fully unique?  : {train['RowId'].is_unique}")
print(f"sample RowIds        : {train['RowId'].head(3).tolist()}")
reconstructed = train["Date"].dt.strftime("%Y%m%d") + "_" + train["SecuritiesCode"].astype(str)
print(f"RowId == 'YYYYMMDD_Code'? : {(reconstructed == train['RowId']).mean():.4%} match")

print("\n" + "=" * 70)
print("5. CARDINALITY OF `SecuritiesCode`")
print("=" * 70)
n_sec = train["SecuritiesCode"].nunique()
print(f"n unique securities in train : {n_sec}")
print(f"min / max code               : {train['SecuritiesCode'].min()} / {train['SecuritiesCode'].max()}")
all_codes = np.sort(train["SecuritiesCode"].unique())
gaps = np.diff(all_codes)
print(f"codes contiguous?            : {(gaps == 1).all()}")
print(f"largest gap between codes    : {int(gaps.max()) if len(gaps) else 0}")
rows_per_sec = train["SecuritiesCode"].value_counts()
print(f"rows per security: min={rows_per_sec.min()}, median={int(rows_per_sec.median())}, max={rows_per_sec.max()}")
print(f"securities with < 30 rows    : {int((rows_per_sec < 30).sum())}")

print("\n" + "=" * 70)
print("6. TRAIN vs TEST — UNSEEN CATEGORIES IN TEST")
print("=" * 70)
train_secs = set(train["SecuritiesCode"].unique())
test_secs  = set(test["SecuritiesCode"].unique())
print(f"Securities in train           : {len(train_secs)}")
print(f"Securities in test            : {len(test_secs)}")
print(f"In test but NOT in train      : {len(test_secs - train_secs)}")
if test_secs - train_secs:
    print(f"  examples: {sorted(test_secs - train_secs)[:10]}")
print(f"In train but NOT in test      : {len(train_secs - test_secs)}")

print("\n" + "=" * 70)
print("7. DATE RANGE & TEMPORAL GRANULARITY")
print("=" * 70)
print(f"train Date range : {train['Date'].min().date()} -> {train['Date'].max().date()}")
print(f"test  Date range : {test['Date'].min().date()}  -> {test['Date'].max().date()}")
print(f"n unique dates   : train={train['Date'].nunique()}, test={test['Date'].nunique()}")
tmp = train.assign(_year=train["Date"].dt.year)
print("trading days per year:")
print(tmp.groupby("_year")["Date"].nunique())
del tmp

print("\n" + "=" * 70)
print("8. DOES `SecuritiesCode` CARRY SIGNAL?")
print("=" * 70)
grp = train.groupby("SecuritiesCode")["Target"].agg(["mean", "std", "count"])
print(f"Overall Target mean/std          : {train['Target'].mean():.6f} / {train['Target'].std():.6f}")
print(f"Per-security Target mean std     : {grp['mean'].std():.6f}")
print(f"Per-security Target mean min/max : {grp['mean'].min():.6f} / {grp['mean'].max():.6f}")

print("\n" + "=" * 70)
print("9. ROW MULTIPLICITY — one row per (Date, Code)?")
print("=" * 70)
dup = train.duplicated(subset=["Date", "SecuritiesCode"]).sum()
print(f"Duplicate (Date, SecuritiesCode) rows in train: {int(dup)}")

print("\n" + "=" * 70)
print("10. NUMERIC COLUMNS — confirm nothing categorical hides here")
print("=" * 70)
num_cols = ["Open", "High", "Low", "Close", "Volume", "Target"]
print(f"All numeric? {all(pd.api.types.is_numeric_dtype(train[c]) for c in num_cols)}")
print(train[num_cols].dtypes)

0. CONFIRM WE'RE ON THE CLEANED DATA
train shape : (1860029, 9)
test shape  : (464894, 9)
train NaN   : 0
test  NaN   : 0

1. COLUMN DTYPES & NON-NULL COUNTS  (train)
                         dtype  n_unique   n_rows  unique_ratio
RowId                      str   1860029  1860029      1.000000
Date            datetime64[us]       968  1860029      0.000520
SecuritiesCode           int64      1997  1860029      0.001074
Open                   float64     21648  1860029      0.011639
High                   float64     22660  1860029      0.012183
Low                    float64     22610  1860029      0.012156
Close                  float64     22699  1860029      0.012204
Volume                   int64     81314  1860029      0.043717
Target                 float64    323943  1860029      0.174160

2. WHICH COLUMNS ARE CATEGORICAL CANDIDATES?
RowId            | dtype=str             | nunique= 1860029 | ratio=1.0000 | dtype=categorical , IDENTIFIER (nearly unique)
Date             | dtyp

In [12]:
from sklearn.model_selection import KFold

# -------- 0. Restart from cleaned frames (not the freq-encoded ones) --------
train = train_clean.copy()
test  = test_clean.copy()
print(f"Before encoding: train={train.shape}  test={test.shape}")

# -------- 1. Drop RowId (100% unique, == Date + '_' + SecuritiesCode) --------
train = train.drop(columns=["RowId"])
test  = test.drop(columns=["RowId"])

# -------- 2. Date -> temporal features --------
for df in (train, test):
    df["year"]       = df["Date"].dt.year
    df["month"]      = df["Date"].dt.month
    df["day"]        = df["Date"].dt.day
    df["dayofweek"]  = df["Date"].dt.dayofweek
    df["quarter"]    = df["Date"].dt.quarter
    df["dayofyear"]  = df["Date"].dt.dayofyear
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)

# -------- 3. SecuritiesCode -> SMOOTHED TARGET ENCODING (leak-free) --------
K = 50                                # smoothing strength
global_mean = train["Target"].mean()  # fallback / prior

# 3a) Out-of-fold encoding for train (each row encoded without its own target)
train = train.sort_values("Date").reset_index(drop=True)
train["SecuritiesCode_te"] = np.nan

kf = KFold(n_splits=5, shuffle=False)     # no shuffle → respects time order
for tr_idx, va_idx in kf.split(train):
    sub = train.iloc[tr_idx]
    agg = sub.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
    smoothed = (agg["count"] * agg["mean"] + K * global_mean) / (agg["count"] + K)
    train.loc[va_idx, "SecuritiesCode_te"] = (
        train.iloc[va_idx]["SecuritiesCode"].map(smoothed).values
    )

# any fold-unseen securities -> global mean
train["SecuritiesCode_te"] = train["SecuritiesCode_te"].fillna(global_mean).astype(np.float32)

# 3b) Full-train mapping for test (unseen → global mean)
agg_full = train.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
smoothed_full = (agg_full["count"] * agg_full["mean"] + K * global_mean) / (agg_full["count"] + K)
test["SecuritiesCode_te"] = (
    test["SecuritiesCode"].map(smoothed_full).fillna(global_mean).astype(np.float32)
)

# -------- 4. Drop raw identifiers --------
train = train.drop(columns=["SecuritiesCode", "Date"])
test  = test.drop(columns=["SecuritiesCode", "Date"])

# -------- 5. Verify --------
print("\n" + "=" * 60)
print("FINAL SHAPES")
print("=" * 60)
print(f"train: {train.shape}")
print(f"test : {test.shape}")
print(f"train NaN: {int(train.isna().sum().sum())}")
print(f"test  NaN: {int(test.isna().sum().sum())}")
print(f"All numeric? {all(pd.api.types.is_numeric_dtype(train[c]) for c in train.columns)}")

print("\nFINAL COLUMNS")
print(list(train.columns))

print("\nSecuritiesCode_te spread (train):")
print(train["SecuritiesCode_te"].describe())
print(f"unique encoded values: {train['SecuritiesCode_te'].nunique()}")

print("\nHEAD")
print(train.head())

Before encoding: train=(1860029, 9)  test=(464894, 9)

FINAL SHAPES
train: (1860029, 14)
test : (464894, 14)
train NaN: 0
test  NaN: 0
All numeric? True

FINAL COLUMNS
['Open', 'High', 'Low', 'Close', 'Volume', 'Target', 'year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'SecuritiesCode_te']

SecuritiesCode_te spread (train):
count    1.860029e+06
mean     3.727229e-04
std      8.624701e-04
min     -8.649977e-03
25%     -1.539882e-04
50%      2.564498e-04
75%      7.536008e-04
max      1.135808e-02
Name: SecuritiesCode_te, dtype: float64
unique encoded values: 9678

HEAD
     Open    High     Low   Close  Volume    Target  year  month  day  \
0  2734.0  2755.0  2730.0  2742.0   31400  0.000730  2017      1    4   
1   329.0   332.0   329.0   332.0   10000 -0.005970  2017      1    4   
2  8330.0  8580.0  8330.0  8580.0   55900 -0.004684  2017      1    4   
3   880.0   907.0   880.0   906.0   37900  0.007609  2017      1    4   
4  3150.0  3210.0  3140.0  3210.0

# Feature Engineering Diagnostic

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# ============================================================
# Rebuild from cleaned frames, keeping Date + SecuritiesCode
# ============================================================
train = train_clean.copy()
test  = test_clean.copy()

K = 50
global_mean = train["Target"].mean()

train = train.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
test  = test.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

# ---- Re-create temporal features (they were lost when resetting) ----
for df in (train, test):
    df["year"]       = df["Date"].dt.year
    df["month"]      = df["Date"].dt.month
    df["day"]        = df["Date"].dt.day
    df["dayofweek"]  = df["Date"].dt.dayofweek
    df["quarter"]    = df["Date"].dt.quarter
    df["dayofyear"]  = df["Date"].dt.dayofyear
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)

# ---- Leak-free out-of-fold target encoding for train ----
train["SecuritiesCode_te"] = np.nan
kf = KFold(n_splits=5, shuffle=False)
for tr_idx, va_idx in kf.split(train):
    sub = train.iloc[tr_idx]
    agg = sub.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
    sm  = (agg["count"] * agg["mean"] + K * global_mean) / (agg["count"] + K)
    train.loc[va_idx, "SecuritiesCode_te"] = (
        train.iloc[va_idx]["SecuritiesCode"].map(sm).values
    )
train["SecuritiesCode_te"] = train["SecuritiesCode_te"].fillna(global_mean).astype(np.float32)

# ---- Full-train mapping for test (unseen → global mean) ----
agg_full = train.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
sm_full  = (agg_full["count"] * agg_full["mean"] + K * global_mean) / (agg_full["count"] + K)
test["SecuritiesCode_te"] = (
    test["SecuritiesCode"].map(sm_full).fillna(global_mean).astype(np.float32)
)

print(f"Rebuilt with keys kept: train={train.shape}  test={test.shape}")
print(f"Columns: {list(train.columns)}")

# ============================================================
# Helpers
# ============================================================
def is_sorted_by(df, keys):
    """Fast lexicographic sort check — casts each column to int64 first."""
    if len(df) < 2:
        return True
    cols = []
    for k in keys:
        s = df[k]
        if pd.api.types.is_datetime64_any_dtype(s):
            cols.append(s.astype("int64").to_numpy())
        else:
            cols.append(s.to_numpy())
    arr = np.column_stack(cols).astype("int64")
    return bool((np.diff(arr, axis=0) >= 0).all())

# ============================================================
# FE DIAGNOSTIC 1 — row order / grouping sanity
# ============================================================
print("\n" + "=" * 70)
print("1. TIME ORDER & GROUPING SANITY")
print("=" * 70)
print(f"train sorted by (Date, Code)? {is_sorted_by(train, ['Date','SecuritiesCode'])}")
print(f"test  sorted by (Date, Code)? {is_sorted_by(test,  ['Date','SecuritiesCode'])}")
print(f"Duplicate (Date, Code) rows in train: "
      f"{int(train.duplicated(['Date','SecuritiesCode']).sum())}")
per_sec_days = train.groupby("SecuritiesCode")["Date"].nunique()
print(f"Days per security: min={per_sec_days.min()}, "
      f"median={int(per_sec_days.median())}, max={per_sec_days.max()}")

# ============================================================
# FE DIAGNOSTIC 2 — return distribution
# ============================================================
print("\n" + "=" * 70)
print("2. RETURN DISTRIBUTION")
print("=" * 70)
train["_ret"] = train.groupby("SecuritiesCode")["Close"].pct_change()
print(train["_ret"].describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]))
print(f"Extreme returns |r|>0.2 : {int((train['_ret'].abs() > 0.2).sum())}")
print(f"Extreme returns |r|>0.5 : {int((train['_ret'].abs() > 0.5).sum())}")

# ============================================================
# FE DIAGNOSTIC 3 — autocorrelation of returns
# ============================================================
print("\n" + "=" * 70)
print("3. AUTOCORRELATION OF RETURNS (lag signal)")
print("=" * 70)
def lag1_autocorr(s):
    s = s.dropna()
    return s.autocorr(lag=1) if len(s) > 3 else np.nan

autocorrs = train.groupby("SecuritiesCode")["_ret"].apply(lag1_autocorr).dropna()
print(f"Mean lag-1 autocorr : {autocorrs.mean():.5f}")
print(f"Std  lag-1 autocorr : {autocorrs.std():.5f}")
print(f"% securities with |autocorr| > 0.05 : "
      f"{(autocorrs.abs() > 0.05).mean()*100:.2f}%")

# ============================================================
# FE DIAGNOSTIC 4 — Volume behaviour
# ============================================================
print("\n" + "=" * 70)
print("4. VOLUME DISTRIBUTION (log transform check)")
print("=" * 70)
print(f"Volume skew (raw)   : {train['Volume'].skew():.3f}")
print(f"Volume skew (log1p) : {np.log1p(train['Volume']).skew():.3f}")
print(f"Volume == 0 count   : {int((train['Volume'] == 0).sum())}")

# ============================================================
# FE DIAGNOSTIC 5 — cross-sectional spread
# ============================================================
print("\n" + "=" * 70)
print("5. CROSS-SECTIONAL VARIATION PER DAY")
print("=" * 70)
daily_std = train.groupby("Date")["_ret"].std()
print(f"Daily cross-sectional return std: mean={daily_std.mean():.5f}, "
      f"median={daily_std.median():.5f}, max={daily_std.max():.5f}")

# ============================================================
# FE DIAGNOSTIC 6 — linear correlation of candidates with Target
# ============================================================
print("\n" + "=" * 70)
print("6. PEARSON CORRELATION OF CANDIDATE FEATURES WITH Target")
print("=" * 70)
train["_hl"]       = (train["High"] - train["Low"]) / train["Close"]
train["_co"]       = (train["Close"] - train["Open"]) / train["Open"]
train["_logvol"]   = np.log1p(train["Volume"])
train["_ret_lag1"] = train.groupby("SecuritiesCode")["_ret"].shift(1)

candidates = ["_hl", "_co", "_logvol", "_ret", "_ret_lag1", "SecuritiesCode_te",
              "year", "month", "dayofweek", "quarter"]
corr = train[candidates + ["Target"]].corr()["Target"].drop("Target")
print(corr.sort_values(ascending=False).round(5))

# Cleanup scratch cols
train = train.drop(columns=["_ret", "_hl", "_co", "_logvol", "_ret_lag1"])
print("\nScratch columns removed. Final train shape:", train.shape)

Rebuilt with keys kept: train=(1860029, 17)  test=(464894, 17)
Columns: ['RowId', 'Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close', 'Volume', 'Target', 'year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'SecuritiesCode_te']

1. TIME ORDER & GROUPING SANITY
train sorted by (Date, Code)? False
test  sorted by (Date, Code)? False
Duplicate (Date, Code) rows in train: 0
Days per security: min=1, median=968, max=968

2. RETURN DISTRIBUTION
count    1.858032e+06
mean     1.081933e-03
std      7.857862e-02
min     -9.004452e-01
0.1%    -1.299276e-01
1%      -6.313131e-02
5%      -3.389831e-02
50%      0.000000e+00
95%      3.569264e-02
99%      7.178608e-02
99.9%    1.664817e-01
max      2.009302e+01
Name: _ret, dtype: float64
Extreme returns |r|>0.2 : 1425
Extreme returns |r|>0.5 : 500

3. AUTOCORRELATION OF RETURNS (lag signal)
Mean lag-1 autocorr : 0.01079
Std  lag-1 autocorr : 0.05467
% securities with |autocorr| > 0.05 : 30.94%

4. VOLUME DISTRIBUTION (log

In [ ]:
# ============================================================
# Build FE on train + test concatenated, so rolling windows
# at test start can use train history (no data loss).
# ============================================================
train = train_clean.copy()
test  = test_clean.copy()

train["_split"] = "train"
test["_split"]  = "test"
full = pd.concat([train, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)
print(f"Concat: {full.shape}")

g = full.groupby("SecuritiesCode", sort=False)

# ---------- Returns (per-security, use past only) ----------
full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

# ---------- Intraday shape (row-wise, no time dependency) ----------
full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

# ---------- Volume ----------
full["log_volume"]   = np.log1p(full["Volume"])
vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

# ---------- Volatility ----------
full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

# ---------- Cross-sectional ranks (within each Date) ----------
full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

# ---------- Drop raw OHLCV (replaced by derived) ----------
full = full.drop(columns=["Open", "High", "Low", "Close", "Volume"])

# ---------- Split back to train / test ----------
train_fe = full[full["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
test_fe  = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

# ---------- Drop rows with NaN at start of each security's history ----------
print(f"Before NaN drop: train={train_fe.shape}  test={test_fe.shape}")
train_fe = train_fe.dropna().reset_index(drop=True)
test_fe  = test_fe.dropna().reset_index(drop=True)
print(f"After  NaN drop: train={train_fe.shape}  test={test_fe.shape}")

# ---------- Drop RowId (final cleanup of identifier) ----------
train_fe = train_fe.drop(columns=["RowId"])
test_fe  = test_fe.drop(columns=["RowId"])

# ============================================================
# Report
# ============================================================
print(f"\nFinal train_fe columns ({len(train_fe.columns)}):")
print(list(train_fe.columns))
print(f"\nNaN in train_fe: {int(train_fe.isna().sum().sum())}")
print(f"NaN in test_fe : {int(test_fe.isna().sum().sum())}")
print(f"\ntrain_fe shape: {train_fe.shape}")
print(f"test_fe  shape: {test_fe.shape}")
print(f"All numeric?   {all(pd.api.types.is_numeric_dtype(train_fe[c]) for c in train_fe.columns)}")

# ============================================================
# Feature-Target correlation (train)
# ============================================================
print("\n" + "=" * 60)
print("FEATURE-TARGET CORRELATION (train)")
print("=" * 60)
feat_cols = [c for c in train_fe.columns if c not in ("Target", "Date")]
corr = (train_fe[feat_cols + ["Target"]]
        .corr()["Target"].drop("Target")
        .sort_values(key=abs, ascending=False))
print(corr.round(5))

print("\nHEAD")
print(train_fe.head())